# Computer Vision · Assignment 2: Hand Gesture Recognition + SOTA synthetic motion
**Team:** Aruzhan Rysbayeva, Alzira Kuanyshbekova, Saltanat Sergeyeva (SE-2403)

Pipeline:
1. A1 dataset with the **same split** as A1 (fair comparison)
2. Synthetic frames from text-to-video clips (Step-Video-T2V idea; see `A2_generate_clips.ipynb`), manual check, split by clip
3. Models: A1 baseline CNN · MobileNetV2 (real only, = A1) · **MobileNetV2 + synthetic (A2)** · ablation: + synthetic without augmentation
4. Evaluation: test metrics, accuracy by **environment** (simulated conditions, held-out synthetic clips, optional new real photos)
5. Grad-CAM explanations · latency · export for the demo app

Runtime → Change runtime type → **GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pillow-heif opencv-python-headless

In [ ]:
import os, json, time, random, shutil, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
register_heif_opener()
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

ROOT      = Path("/content/drive/MyDrive/comp-vis1")
REAL_DIR  = ROOT / "hand_gestures"             # A1 dataset
A2        = ROOT / "a2"
CLIPS_DIR = A2 / "synthetic_clips"             # from A2_generate_clips.ipynb
SYN_DIR   = A2 / "synthetic_frames"            # extracted frames (created below)
ENV_DIR   = A2 / "test_environments"           # OPTIONAL new real photos: <env>/<class>/*.jpg
OUT       = A2 / "outputs"; FIG = OUT / "figures"
for d in (SYN_DIR, FIG): d.mkdir(parents=True, exist_ok=True)

SEED = 42
EXT = {".jpg", ".jpeg", ".png", ".heic"}
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("Real data exists:", REAL_DIR.exists(), "| clips exist:", CLIPS_DIR.exists())

### Shared code (identical to `app/gesture_utils.py`)

In [ ]:
"""Shared code for the A2 notebook and the demo app.

Model input: RGB float32 images in [0, 1], shape (224, 224, 3).
The model rescales internally to [-1, 1] (MobileNetV2 preprocessing).
"""
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models

CLASSES = ["peace-sign", "thumbs-up", "stop"]
IMG_SIZE = 224


# ----------------------------------------------------------------- model
def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal", name="aug_flip"),
        layers.RandomRotation(0.04, name="aug_rot"),
        layers.RandomZoom(0.1, name="aug_zoom"),
        layers.RandomTranslation(0.1, 0.1, name="aug_shift"),
        layers.RandomContrast(0.1, name="aug_contrast"),
        layers.RandomBrightness(0.2, value_range=(0, 1), name="aug_bright"),
    ], name="augment")


def build_mobilenet(use_aug=True, imagenet=True, dropout=0.2, n_classes=3):
    """MobileNetV2 transfer model (same head as A1). All layers have fixed names."""
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = build_augmentation()(inp) if use_aug else inp
    x = layers.Rescaling(2.0, offset=-1.0, name="rescale")(x)   # [0,1] -> [-1,1]
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
        weights="imagenet" if imagenet else None, name="mobilenetv2")
    base.trainable = False
    x = base(x, training=False)          # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    out = layers.Dense(n_classes, activation="softmax", name="pred")(x)
    return models.Model(inp, out, name="gesture_mobilenetv2")


def build_baseline_cnn(n_classes=3):
    """Simple CNN baseline from A1."""
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = build_augmentation()(inp)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return models.Model(inp, out, name="baseline_cnn")


# ------------------------------------------------------------ preprocessing
def to_model_input(img):
    """PIL image or HxWx3 uint8/float array -> (224,224,3) float32 in [0,1]."""
    arr = np.asarray(img.convert("RGB") if hasattr(img, "convert") else img)
    if arr.dtype != np.uint8:
        arr = np.clip(arr * (255.0 if arr.max() <= 1.0 else 1.0), 0, 255).astype(np.uint8)
    arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return arr.astype(np.float32) / 255.0


# ------------------------------------------------------- stress conditions
def _rng(seed):
    return np.random.default_rng(seed)


def perturb(img01, condition, seed=0):
    """Apply a simulated environment condition to a [0,1] image (H,W,3)."""
    x = img01.astype(np.float32)
    h, w = x.shape[:2]
    if condition == "clean":
        return x
    if condition == "dark":                       # low exposure
        return np.clip((x ** 1.8) * 0.45, 0, 1)
    if condition == "overexposed":
        return np.clip(x * 1.7 + 0.15, 0, 1)
    if condition == "motion_blur":
        k = np.zeros((15, 15), np.float32)
        k[7, :] = 1.0 / 15
        return cv2.filter2D(x, -1, k)
    if condition == "noise":
        return np.clip(x + _rng(seed).normal(0, 0.08, x.shape), 0, 1).astype(np.float32)
    if condition == "low_res":
        small = cv2.resize(x, (w // 4, h // 4), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    if condition == "rotated":
        m = cv2.getRotationMatrix2D((w / 2, h / 2), 25, 1.0)
        return cv2.warpAffine(x, m, (w, h), borderMode=cv2.BORDER_REFLECT)
    if condition == "occlusion":
        r = _rng(seed)
        ph, pw = int(h * 0.3), int(w * 0.3)
        y0, x0 = r.integers(0, h - ph), r.integers(0, w - pw)
        x = x.copy()
        x[y0:y0 + ph, x0:x0 + pw] = 0
        return x
    raise ValueError(condition)


CONDITIONS = ["clean", "dark", "overexposed", "motion_blur",
              "noise", "low_res", "rotated", "occlusion"]


# ---------------------------------------------------------------- Grad-CAM
def grad_cam(model, img01, class_idx=None):
    """Grad-CAM on the last conv block of MobileNetV2 (layer 'out_relu').

    Returns (heatmap HxW in [0,1] at 224x224, probs, class_idx).
    """
    base = model.get_layer("mobilenetv2")
    rescale = model.get_layer("rescale")
    gap = model.get_layer("gap")
    pred = model.get_layer("pred")
    x = tf.convert_to_tensor(img01[None].astype(np.float32))
    with tf.GradientTape() as tape:
        conv = base(rescale(x), training=False)
        tape.watch(conv)
        logits = tf.matmul(gap(conv), pred.kernel) + pred.bias
        probs = tf.nn.softmax(logits)
        if class_idx is None:
            class_idx = int(tf.argmax(probs[0]))
        score = logits[:, class_idx]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(conv * weights, axis=-1))[0].numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, probs[0].numpy(), class_idx


def overlay_cam(img01, cam, alpha=0.45):
    """Return uint8 RGB image with a JET heatmap overlay."""
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    out = (1 - alpha) * img01 + alpha * heat
    return np.uint8(np.clip(out, 0, 1) * 255)


# ------------------------------------------------------- synthetic labeling
def watermark_frame(frame_rgb_uint8, text="SYNTHETIC - AI-generated"):
    """Burn a visible 'synthetic' label into a frame (for display only, never for training)."""
    f = frame_rgb_uint8.copy()
    h, w = f.shape[:2]
    scale = max(0.5, w / 900)
    th = int(34 * scale)
    cv2.rectangle(f, (0, 0), (w, th), (0, 0, 0), -1)
    cv2.putText(f, text, (8, int(th * 0.72)), cv2.FONT_HERSHEY_SIMPLEX,
                0.8 * scale, (255, 220, 0), max(1, int(2 * scale)), cv2.LINE_AA)
    return f


def read_video_frames(path, n=None, every=1):
    """Read RGB uint8 frames from a video. n = evenly spaced count (optional)."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    if n is not None and len(frames) > n:
        idx = np.linspace(0, len(frames) - 1, n).round().astype(int)
        frames = [frames[i] for i in idx]
    return frames[::every]

## 1. A1 dataset, same split as A1

In [ ]:
image_paths, labels = [], []
for label, c in enumerate(CLASSES):
    for f in sorted((REAL_DIR / c).iterdir()):
        if f.is_file() and f.suffix.lower() in EXT:
            image_paths.append(str(f)); labels.append(label)

X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.30, random_state=SEED, stratify=labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)
print("Train/Val/Test:", len(X_train), len(X_val), len(X_test))

def load_images(paths):
    out = []
    for p in paths:
        img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        out.append(to_model_input(img))
    return np.stack(out).astype(np.float32)

Xtr, Xva, Xte = load_images(X_train), load_images(X_val), load_images(X_test)
ytr, yva, yte = map(np.array, (y_train, y_val, y_test))
print(Xtr.shape, Xva.shape, Xte.shape)

## 2. Synthetic frames from generated clips
* From each clip we take **8 evenly spaced frames**, resized to 224×224.
* **Split by clip** (never by frame): 75% of clips → training, 25% → *synthetic stress-test set*.
* Frames used for training are **not watermarked** (a watermark would be a shortcut feature). They are labelled by folder/file name and in the report; every synthetic image shown to people is watermarked.

In [ ]:
FRAMES_PER_CLIP = 8
rows = []
for c in CLASSES:
    for clip in sorted((CLIPS_DIR / c).glob("*.mp4")):
        parts = clip.stem.split("__")                       # cls__env__camera__idx
        env, cam = (parts[1], parts[2]) if len(parts) >= 4 else ("unknown", "unknown")
        frames = read_video_frames(clip, n=FRAMES_PER_CLIP)
        for k, fr in enumerate(frames):
            dst = SYN_DIR / c / f"{clip.stem}__f{k}.jpg"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                Image.fromarray(fr).resize((224, 224)).save(dst, quality=95)
            rows.append(dict(path=str(dst), cls=c, label=CLASSES.index(c),
                             clip=clip.stem, env=env, camera=cam))
syn = pd.DataFrame(rows)
print(syn.groupby("cls").agg(clips=("clip", "nunique"), frames=("path", "count")))

### 2.1 Manual check. ⚠️ Required step
Look at the contact sheet. Write the names of clips where the gesture is wrong (wrong fingers, deformed hand, no hand) into `REJECT_CLIPS` and run the next cell.

In [ ]:
def contact_sheet(df, cls, n_cols=6):
    clips = df[df.cls == cls]["clip"].unique()
    fig, axes = plt.subplots(int(np.ceil(len(clips) / n_cols)), n_cols,
                             figsize=(n_cols * 2.6, np.ceil(len(clips) / n_cols) * 2.9), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, clip in zip(axes.flat, clips):
        p = df[df["clip"] == clip].path.iloc[FRAMES_PER_CLIP // 2]
        ax.imshow(Image.open(p)); ax.set_title(clip.split("__", 1)[1][:28], fontsize=7)
    fig.suptitle(f"{cls}: SYNTHETIC frames (middle frame of each clip)"); plt.tight_layout(); plt.show()

for c in CLASSES:
    contact_sheet(syn, c)

In [ ]:
REJECT_CLIPS = [
    # "thumbs-up__dark_room__pan__03",
]
syn["kept"] = ~syn["clip"].isin(REJECT_CLIPS)
print("Rejected clips:", len(REJECT_CLIPS), "| kept frames:", syn.kept.sum(), "of", len(syn))
syn = syn[syn.kept].reset_index(drop=True)

# split by clip, stratified by class
clips = syn.drop_duplicates("clip")[["clip", "label"]]
tr_clips, st_clips = train_test_split(clips["clip"], test_size=0.25, random_state=SEED,
                                      stratify=clips.label)
syn["split"] = np.where(syn["clip"].isin(tr_clips), "train", "stress")
syn.to_csv(OUT / "synthetic_frames.csv", index=False)

Xsyn = load_images(syn[syn.split == "train"].path); ysyn = syn[syn.split == "train"].label.values
Xstr = load_images(syn[syn.split == "stress"].path); ystr = syn[syn.split == "stress"].label.values
print("Synthetic train frames:", len(Xsyn), "| synthetic stress frames:", len(Xstr))

In [ ]:
# Dataset summary table for the report
summary = pd.DataFrame({
    "Real train": pd.Series(ytr).value_counts().sort_index().values,
    "Real val":   pd.Series(yva).value_counts().sort_index().values,
    "Real test":  pd.Series(yte).value_counts().sort_index().values,
    "Synthetic train": [int((ysyn == i).sum()) for i in range(3)],
    "Synthetic stress": [int((ystr == i).sum()) for i in range(3)],
}, index=CLASSES)
summary.loc["Total"] = summary.sum()
syn_share = len(Xsyn) / (len(Xsyn) + len(Xtr))
print(summary.to_markdown())
print(f"\nSynthetic share of the training set: {syn_share:.1%}")

## 3. Training
Same recipe as A1 for every MobileNetV2 variant: stage 1 (frozen base, 10 epochs, Adam) → stage 2 (last 30 layers unfrozen, Adam 1e-5, up to 15 epochs, early stopping on val loss, patience 4).
Validation is always the **real** validation set.

In [ ]:
def train_mobilenet(Xa, ya, use_aug=True, name="model"):
    tf.keras.utils.set_random_seed(SEED)
    m = build_mobilenet(use_aug=use_aug)
    m.compile(optimizer=tf.keras.optimizers.Adam(), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    h1 = m.fit(Xa, ya, validation_data=(Xva, yva), epochs=10, batch_size=16, verbose=2)
    base = m.get_layer("mobilenetv2"); base.trainable = True
    for l in base.layers[:-30]: l.trainable = False
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    h2 = m.fit(Xa, ya, validation_data=(Xva, yva), epochs=15, batch_size=16, callbacks=[es], verbose=2)
    hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
    hist["stage1_epochs"] = len(h1.history["loss"])
    m.save(OUT / f"{name}.keras")
    return m, hist

def train_baseline():
    tf.keras.utils.set_random_seed(SEED)
    m = build_baseline_cnn()
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=15, batch_size=16, verbose=2)
    return m, dict(h.history)

Xmix = np.concatenate([Xtr, Xsyn]); ymix = np.concatenate([ytr, ysyn])
models_, hists = {}, {}
models_["Baseline CNN (A1)"], hists["Baseline CNN (A1)"] = train_baseline()
models_["MobileNetV2 real (A1)"], hists["MobileNetV2 real (A1)"] = train_mobilenet(Xtr, ytr, True, "mnv2_real")
models_["MobileNetV2 + synthetic (A2)"], hists["MobileNetV2 + synthetic (A2)"] = train_mobilenet(Xmix, ymix, True, "mnv2_syn")
# Ablation: same data, augmentation switched off
models_["Ablation: + synthetic, no aug"], hists["Ablation: + synthetic, no aug"] = train_mobilenet(Xmix, ymix, False, "mnv2_syn_noaug")

In [ ]:
# Training curves
fig, axes = plt.subplots(2, len(hists), figsize=(4.2 * len(hists), 7))
for j, (name, h) in enumerate(hists.items()):
    for i, (k, t) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
        ax = axes[i, j]
        ax.plot(h[k], label="train"); ax.plot(h["val_" + k], label="val")
        if "stage1_epochs" in h: ax.axvline(h["stage1_epochs"] - 0.5, ls="--", c="gray", lw=1)
        ax.set_title(f"{name}\n{t}", fontsize=9); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "training_curves.png", dpi=150); plt.show()

## 4. Evaluation on the real test set (97 images)

In [ ]:
def predict(m, X, bs=32):
    return m.predict(X, batch_size=bs, verbose=0).argmax(1)

def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return dict(Accuracy=accuracy_score(y, p), Precision=pr, Recall=rc, **{"Macro-F1": f1})

preds = {n: predict(m, Xte) for n, m in models_.items()}
main_tab = pd.DataFrame({n: metrics(yte, p) for n, p in preds.items()}).T.round(3)
print(main_tab.to_markdown())

fig, axes = plt.subplots(1, len(preds), figsize=(4.3 * len(preds), 3.8))
for ax, (n, p) in zip(axes, preds.items()):
    sns.heatmap(confusion_matrix(yte, p), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(n, fontsize=9); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrices.png", dpi=150); plt.show()

for n in ["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]:
    print(n); print(classification_report(yte, preds[n], target_names=CLASSES, digits=3))

## 5. Accuracy by environment
**5a. Simulated conditions** on the real test set (dark, overexposed, motion blur, noise, low resolution, rotation, occlusion).
**5b. Held-out synthetic clips** (stress-test generator), grouped by the environment in the prompt.
**5c. New real photos** per environment, if `test_environments/` exists (recommended: 20–30 photos per class per environment).

In [ ]:
cond_rows = {}
for n, m in models_.items():
    cond_rows[n] = {}
    for c in CONDITIONS:
        Xc = np.stack([perturb(x, c, seed=i) for i, x in enumerate(Xte)])
        cond_rows[n][c] = accuracy_score(yte, predict(m, Xc))
cond_tab = pd.DataFrame(cond_rows).T.round(3)
print(cond_tab.to_markdown())

plt.figure(figsize=(11, 3.2))
sns.heatmap(cond_tab, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.3, vmax=1)
plt.title("Accuracy under simulated conditions (real test set)")
plt.tight_layout(); plt.savefig(FIG / "acc_by_condition.png", dpi=150); plt.show()

# example of every condition
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(2.2 * len(CONDITIONS), 2.6))
for ax, c in zip(axes, CONDITIONS):
    ax.imshow(perturb(Xte[0], c)); ax.set_title(c, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "conditions_examples.png", dpi=150); plt.show()

In [ ]:
stress = syn[syn.split == "stress"].copy()
for n in ["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]:
    stress[n] = predict(models_[n], Xstr) == ystr
syn_env_tab = stress.groupby("env")[["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]].mean().round(3)
syn_env_tab.loc["ALL"] = stress[["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]].mean().round(3)
syn_env_tab["frames"] = stress.groupby("env").size().reindex(syn_env_tab.index).fillna(len(stress)).astype(int)
print("Per-frame accuracy on held-out SYNTHETIC clips:")
print(syn_env_tab.to_markdown())

# clip-level majority vote vs single frame
vote = stress.groupby("clip").apply(lambda d: pd.Series({
    "single-frame acc": d["MobileNetV2 + synthetic (A2)"].mean(),
    "clip vote correct": float(np.bincount(predict(models_["MobileNetV2 + synthetic (A2)"],
                          load_images(d.path)), minlength=3).argmax() == d.label.iloc[0])}))
print("\nClip-level vote accuracy:", vote["clip vote correct"].mean().round(3))

In [ ]:
real_env_tab = None
if ENV_DIR.exists():
    rows = []
    for env in sorted(p.name for p in ENV_DIR.iterdir() if p.is_dir()):
        paths, ys = [], []
        for i, c in enumerate(CLASSES):
            for f in sorted((ENV_DIR / env / c).glob("*")):
                if f.suffix.lower() in EXT: paths.append(str(f)); ys.append(i)
        if not paths: continue
        Xe = load_images(paths); ye = np.array(ys)
        r = {"environment": env, "images": len(ye)}
        for n, m in models_.items(): r[n] = accuracy_score(ye, predict(m, Xe))
        rows.append(r)
    real_env_tab = pd.DataFrame(rows).set_index("environment").round(3)
    print(real_env_tab.to_markdown())
else:
    print("No test_environments/ folder: skipped (optional but recommended).")

## 6. Explainability: Grad-CAM

In [ ]:
def cam_grid(idx_list, fname, title):
    names = ["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]
    fig, axes = plt.subplots(len(idx_list), 3, figsize=(8, 2.8 * len(idx_list)), squeeze=False)
    for r, i in enumerate(idx_list):
        axes[r, 0].imshow(Xte[i]); axes[r, 0].set_title(f"true: {CLASSES[yte[i]]}", fontsize=9)
        for k, n in enumerate(names):
            cam, probs, ci = grad_cam(models_[n], Xte[i])
            axes[r, k + 1].imshow(overlay_cam(Xte[i], cam))
            axes[r, k + 1].set_title(f"{n.split(' (')[0]}\n{CLASSES[ci]} {probs[ci]:.0%}", fontsize=8)
        for a in axes[r]: a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG / fname, dpi=150); plt.show()

rng = np.random.default_rng(SEED)
examples = [int(rng.choice(np.where(yte == c)[0])) for c in range(3)]
cam_grid(examples, "gradcam_examples.png", "Grad-CAM: one test image per class")

wrong = np.where(preds["MobileNetV2 + synthetic (A2)"] != yte)[0][:4]
if len(wrong): cam_grid(list(wrong), "gradcam_errors.png", "Grad-CAM on misclassified test images")

# Grad-CAM on a dark image
cam, probs, ci = grad_cam(models_["MobileNetV2 + synthetic (A2)"], perturb(Xte[examples[0]], "dark"))
plt.figure(figsize=(3, 3)); plt.imshow(overlay_cam(perturb(Xte[examples[0]], "dark"), cam))
plt.title(f"dark: {CLASSES[ci]} {probs[ci]:.0%}"); plt.axis("off"); plt.show()

## 7. Latency

In [ ]:
def latency(m, device, n=100):
    x = Xte[:1]
    with tf.device(device):
        for _ in range(10): m(x, training=False)        # warm-up
        ts = []
        for _ in range(n):
            t0 = time.perf_counter(); m(x, training=False).numpy(); ts.append((time.perf_counter() - t0) * 1000)
    return np.mean(ts), np.percentile(ts, 95)

lat_rows = []
devices = ["/CPU:0"] + (["/GPU:0"] if tf.config.list_physical_devices("GPU") else [])
for n in ["Baseline CNN (A1)", "MobileNetV2 + synthetic (A2)"]:
    for d in devices:
        mean, p95 = latency(models_[n], d)
        lat_rows.append({"Model": n, "Device": d.strip("/:0"), "Mean ms": round(mean, 1),
                         "p95 ms": round(p95, 1), "FPS": round(1000 / mean, 1),
                         "Params": models_[n].count_params()})
# Grad-CAM cost
t0 = time.perf_counter()
for i in range(20): grad_cam(models_["MobileNetV2 + synthetic (A2)"], Xte[i])
gc_ms = (time.perf_counter() - t0) / 20 * 1000
lat_tab = pd.DataFrame(lat_rows)
print(lat_tab.to_markdown(index=False))
print(f"Grad-CAM (prediction + explanation): {gc_ms:.1f} ms per image")
print("Environment:", platform.platform(), "| CPU:", platform.processor() or "n/a",
      "| GPU:", tf.config.list_physical_devices("GPU"))

## 8. Save results and export for the app

In [ ]:
# choose the deployed model by VALIDATION macro-F1 (the test set is not used for model choice)
val_f1 = {n: metrics(yva, predict(models_[n], Xva))["Macro-F1"]
          for n in ["MobileNetV2 real (A1)", "MobileNetV2 + synthetic (A2)"]}
best = max(val_f1, key=val_f1.get)
print("Validation macro-F1:", val_f1, "→ deploy:", best)

APP = OUT / "app_export"; (APP / "examples" / "synthetic").mkdir(parents=True, exist_ok=True)
models_[best].save(APP / "gesture_model.keras")

# a few held-out synthetic clips for the app's "Synthetic clips" tab, watermarked
import cv2
for clip in stress.drop_duplicates("clip").groupby("cls").head(3)["clip"]:
    cls = clip.split("__")[0]
    frames = read_video_frames(CLIPS_DIR / cls / f"{clip}.mp4")
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(str(APP / "examples" / "synthetic" / f"{clip}.mp4"),
                         cv2.VideoWriter_fourcc(*"mp4v"), 16, (w, h))
    for fr in frames: vw.write(cv2.cvtColor(watermark_frame(fr), cv2.COLOR_RGB2BGR))
    vw.release()

md = ["## Results (test set, 97 real images)", main_tab.to_markdown(), "",
      "## Accuracy under simulated conditions", cond_tab.to_markdown(), "",
      "## Held-out synthetic clips (per-frame accuracy)", syn_env_tab.to_markdown(), "",
      "## Latency", lat_tab.to_markdown(index=False), f"\nGrad-CAM: {gc_ms:.1f} ms/image"]
if real_env_tab is not None:
    md += ["", "## New real photos by environment", real_env_tab.to_markdown()]
md += ["", "## Dataset", summary.to_markdown(), f"\nSynthetic share of training set: {syn_share:.1%}"]
(OUT / "results.md").write_text("\n".join(md)); shutil.copy(OUT / "results.md", APP / "results.md")
main_tab.to_csv(OUT / "main_metrics.csv"); cond_tab.to_csv(OUT / "acc_by_condition.csv")
json.dump({k: {kk: (vv if isinstance(vv, int) else [float(x) for x in vv]) for kk, vv in h.items()}
           for k, h in hists.items()}, open(OUT / "histories.json", "w"))
print("\n".join(md))
print("\nSaved to", OUT, "→ copy app_export/* into the app/ folder of the repo.")

In [ ]:
# download the app export as a zip
shutil.make_archive("/content/app_export", "zip", APP)
from google.colab import files
files.download("/content/app_export.zip")